# 토스 경진대회 - 클릭 예측 (LightGBM + CV + p-value 피처 셀렉션)

**홍익대 3학년 | 데이터사이언스 | AI 해커톤 준비 중**

---
## 핵심 전략
1. **Resampling**: 5-fold CV + `scale_pos_weight` (SMOTE 제거)
2. **범주형 피처**: One-hot → Logistic → **p-value < 0.05 필터링**
3. **부스팅 모델**: LightGBM + early stopping
4. **검증**: CV 기반 OOF AUC

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 디렉토리 설정
notebook_dir = r'C:\Users\tkdwl\Desktop\토스 경진대회'
os.chdir(notebook_dir)

# 데이터 로드
df_train = pd.read_parquet('./train.parquet')
df_test = pd.read_parquet('./test.parquet')

print(f"Train: {df_train.shape} | Test: {df_test.shape}")

## 1. 결측값 처리

In [2]:
target_col = 'clicked'
id_col = 'ID'
col_drop_threshold = 0.05

# 5% 이상 결측 컬럼 제거
missing_train = df_train.isnull().mean()
missing_test = df_test.isnull().mean()
high_missing = (missing_train >= col_drop_threshold) | (missing_test >= col_drop_threshold)
high_missing_cols = high_missing[high_missing].index.tolist()
high_missing_cols = [c for c in high_missing_cols if c not in [target_col, id_col]]

if high_missing_cols:
    df_train = df_train.drop(columns=high_missing_cols)
    df_test = df_test.drop(columns=high_missing_cols)
    print(f"제거된 컬럼: {len(high_missing_cols)}개")

# 수치형/범주형 분리
num_cols = df_train.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]
cat_cols = df_train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in [id_col]]

# 수치형 결측 행 제거
df_train = df_train.dropna(subset=num_cols).reset_index(drop=True)
df_test = df_test.dropna(subset=num_cols).reset_index(drop=True)

# 범주형 최빈값 대체
from sklearn.impute import SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
df_train[cat_cols] = cat_imputer.fit_transform(df_train[cat_cols])
df_test[cat_cols] = cat_imputer.transform(df_test[cat_cols])

print(f"결측 처리 완료: train={df_train.isnull().sum().sum()}, test={df_test.isnull().sum().sum()}")

## 2. 범주형 피처: p-value 기반 필터링 (Classification 슬라이드 적용)

In [3]:
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

cat_for_selection = ['gender', 'age_group', 'inventory_id', 'day_of_week']
X_cat = df_train[cat_for_selection].astype(str)
y = df_train[target_col]

# One-hot encoding
ohe = OneHotEncoder(sparse=True, handle_unknown='ignore')
X_ohe = ohe.fit_transform(X_cat)
feature_names = ohe.get_feature_names_out(cat_for_selection)

# Logistic Regression + p-value
X_const = sm.add_constant(X_ohe)
logit = sm.Logit(y, X_const)
result = logit.fit_regularized(method='l1', alpha=0.01, disp=False)

p_vals = result.pvalues[1:]  # 상수항 제외
significant = p_vals < 0.05
selected_features = feature_names[significant]

print(f"유의미한 범주형 피처: {len(selected_features)}개 / {len(feature_names)}개")

# 선택된 피처만 one-hot으로 변환
ohe_selected = OneHotEncoder(sparse=False)
ohe_selected.categories_ = ohe.categories_

train_selected = pd.DataFrame(
    ohe_selected.fit_transform(X_cat),
    columns=feature_names
)[selected_features]

test_selected = pd.DataFrame(
    ohe_selected.transform(df_test[cat_for_selection].astype(str)),
    columns=feature_names
)[selected_features]

# 원본 df에 추가 + 기존 cat 제거
for col in selected_features:
    df_train[col] = train_selected[col].values
    df_test[col] = test_selected[col].values

df_train = df_train.drop(columns=cat_for_selection)
df_test = df_test.drop(columns=cat_for_selection)

print(f"피처 엔지니어링 전 피처 수: {df_train.shape[1] - 2}")

## 3. 피처 엔지니어링

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# hour 처리
df_train['hour'] = pd.to_numeric(df_train['hour'], errors='coerce').fillna(12)
df_test['hour'] = pd.to_numeric(df_test['hour'], errors='coerce').fillna(12)

for df in [df_train, df_test]:
    df['is_weekend'] = df['day_of_week'].astype(str).str.contains('5|6').astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# seq 피처
def process_seq(df, top_n=20):
    df['seq_length'] = df['seq'].str.split(',').str.len()
    df['seq_unique'] = df['seq'].str.split(',').apply(lambda x: len(set(x)) if isinstance(x, str) else 0)
    df['seq_diversity'] = df['seq_unique'] / (df['seq_length'] + 1e-8)
    items = [i.strip() for s in df['seq'].str.split(',') for i in s]
    top_items = pd.Series(items).value_counts().head(100).index
    for item in top_items[:top_n]:
        df[f'seq_has_{item}'] = df['seq'].str.contains(item, regex=False).astype(int)
    return df

df_train = process_seq(df_train)
df_test = process_seq(df_test)

# 클러스터링
cluster_cols = [c for c in num_cols if c in df_train.columns]
scaler = StandardScaler()
scaled_train = scaler.fit_transform(df_train[cluster_cols])
scaled_test = scaler.transform(df_test[cluster_cols])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_train['cluster'] = kmeans.fit_predict(scaled_train)
df_test['cluster'] = kmeans.predict(scaled_test)

# 클러스터 one-hot
df_train = pd.get_dummies(df_train, columns=['cluster'], prefix='cluster')
df_test = pd.get_dummies(df_test, columns=['cluster'], prefix='cluster')
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# 수치형 스케일링
num_cols_final = [c for c in num_cols if c in df_train.columns]
df_train[num_cols_final] = scaler.fit_transform(df_train[num_cols_final])
df_test[num_cols_final] = scaler.transform(df_test[num_cols_final])

# age_click_mean
age_click = df_train.groupby('age_group')[target_col].mean()
df_train['age_click_mean'] = df_train['age_group'].map(age_click)
df_test['age_click_mean'] = df_test['age_group'].map(age_click).fillna(df_train[target_col].mean())

print(f"최종 피처 수: {df_train.shape[1] - 2}")

## 4. 5-fold CV + LightGBM (Resampling 슬라이드 적용)

In [5]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import gc

feature_cols = [c for c in df_train.columns if c not in [target_col, id_col]]
X = df_train[feature_cols]
y = df_train[target_col]

# scale_pos_weight
neg, pos = y.value_counts()
scale_pos_weight = neg / pos
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

# 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(df_test))
cv_scores = []

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'scale_pos_weight': scale_pos_weight,
    'verbose': -1,
    'seed': 42
}

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold+1}/5")
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_trn, y_trn)
    lgb_valid = lgb.Dataset(X_val, y_val, reference=lgb_train)

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_valid],
        early_stopping_rounds=200,
        verbose_eval=500
    )

    oof_preds[val_idx] = model.predict(X_val)
    auc = roc_auc_score(y_val, oof_preds[val_idx])
    cv_scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.5f}")

    test_preds += model.predict(df_test[feature_cols]) / skf.n_splits

    del model, X_trn, X_val
    gc.collect()

print(f"\nCV AUC: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}")

## 5. 제출 파일 생성

In [6]:
submission = pd.DataFrame({
    'ID': df_test[id_col],
    'clicked': test_preds
})
submission.to_csv('submission_final.csv', index=False)
print("제출 완료: submission_final.csv")